# P4 — Transfer Learning & Fine-Tuning (CPU-Friendly)
## Data Set: Oxford-IIIT Pet
- #### Feature Extraction vs Fine-Tuning | Performance Comparison
### Models: VGG16 • ResNet50 • DenseNet121
#### Solution Notebook

**Hardware assumption:** CPU-only laptops/PC are acceptable (training time may vary).  
**Dataset:** Oxford-IIIT Pet (37 classes)  
**Recommended settings:** `IMG_SIZE=(128,128)`, `BATCH_SIZE=32`, `EPOCHS=5-6` (CPU-friendly)

---

This solution notebook provides a clean reference implementation for:
- Building a reproducible `tf.data` pipeline
- Training **frozen-backbone** models (feature extraction)
- Running **one controlled fine-tuning** experiment
- Comparing results (accuracy, training time, parameter counts)


## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [3]:
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from tensorflow.keras.applications import resnet, vgg16, densenet

print("TensorFlow:", tf.__version__)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

TensorFlow: 2.20.0


## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q9)** and continues by answering questions in the **Markdown cells (Q10–Q13)**. These correspond to the questions listed in the assignment description on Canvas. Complete each cell by following the instructions provided in the **preceding Markdown cells**.

Tasks:
- This assignment focuses on **transfer learning** with pretrained CNN backbones.
- You will train **three frozen-backbone models** for a fair comparison:
  - **VGG16** (frozen base)
  - **ResNet50** (frozen base)
  - **DenseNet121** (frozen base)
- Then run **one controlled fine-tuning experiment** (unfreeze a small portion of one backbone with a smaller learning rate).
- Keep your training CPU-feasible (use the recommended settings unless you have a GPU).

## Q1 — Load Dataset & Inspect
Use TFDS: `oxford_iiit_pet` and inspect shapes/classes.
### Student Tasks

- Load the Oxford-IIIT Pet dataset and split into ds_train (80%), ds_val (20%), and ds_test.

- Extract number of classes and class names from ds_info.

- Display one sample: image shape, label index, and class name.

In [4]:
#============================================================
#Question Q1 — Load Oxford-IIIT Pet Dataset (Fill in the Blanks)
#============================================================
#Complete the TODO sections to:
#1) Load the Oxford-IIIT Pet dataset using TensorFlow Datasets
#2) Create train/validation/test splits
#3) Extract class names and number of classes
#4) Print dataset information
#5) Inspect one example image and label
#============================================================

# TODO 1: Load dataset with train/validation/test splits
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train[:80%]", "train[80%:]", "test"],
    as_supervised=True,
    with_info=True,
)

# TODO 2: Get number of classes
num_classes = ds_info.features["label"].num_classes

# TODO 3: Get class names
class_names = ds_info.features["label"].names

# TODO 4: Print dataset information
print("Num classes:", num_classes)
print("Example classes:", class_names[:5])

# TODO 5: Inspect one example from the training set
for x, y in ds_train.take(1):
    print("Raw image shape:", x.shape,
          "| label:", int(y),
          "| class:", class_names[int(y)])

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.HBRPOI_4.0.0/oxford_iiit_pet-train.tfrecord-[0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.HBRPOI_4.0.0/oxford_iiit_pet-test.tfrecord-[0-9…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.
Num classes: 37
Example classes: ['Abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle']
Raw image shape: (500, 500, 3) | label: 33 | class: Sphynx


## Q2 — Preprocessing & `tf.data` Pipeline
Resize to **128×128**, normalize, add light augmentation for training.

### Student Tasks

- Set image size and batch size.

- Preprocess images by resizing and normalizing.

- Apply data augmentation to the training dataset.

- Create batched and prefetched train, validation, and test pipelines.

In [5]:
#============================================================
#Question Q2 — Image Preprocessing & Data Pipeline (Fill in the Blanks)
#============================================================
#Complete the TODO sections to:
#1) Define image size (128×128) and batch size (32 to 64)
#2) Implement preprocessing (resize + normalization)
#3) Implement simple data augmentation
#4) Build TensorFlow data pipelines
#5) Shuffle, batch, and prefetch the datasets
#============================================================

# TODO 1: Define image size and batch size
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# ------------------------------------------------------------
# TODO 2: Preprocessing function
# Resize images and normalize pixel values to [0,1]
# ------------------------------------------------------------
@tf.function
def preprocess(image, label):

    # Resize image
    image = tf.image.resize(image, IMG_SIZE)

    # Convert to float32
    image = tf.cast(image, tf.float32)

    # Normalize pixels
    image = image / 225.0

    return image, label

# ------------------------------------------------------------
# TODO 3: Data augmentation function
# ------------------------------------------------------------
@tf.function
def augment(image, label):

    # Random horizontal flip
    image = tf.image.random_flip_left_right(image)

    # Random brightness
    image = tf.image.random_brightness(image, max_delta=0.1)

    return image, label

# ------------------------------------------------------------
# TODO 4: Apply preprocessing and augmentation
# ------------------------------------------------------------
train_ds = ds_train.map(preprocess, num_parallel_calls=AUTOTUNE)\
                   .map(augment, num_parallel_calls=AUTOTUNE)

val_ds   = ds_val.map(preprocess, num_parallel_calls=AUTOTUNE)

test_ds  = ds_test.map(preprocess, num_parallel_calls=AUTOTUNE)

# ------------------------------------------------------------
# TODO 5: Shuffle, batch, and prefetch
# ------------------------------------------------------------
train_ds = train_ds.shuffle(1000, seed=SEED)\
                   .batch(BATCH_SIZE)\
                   .prefetch(AUTOTUNE)

val_ds   = val_ds.batch(BATCH_SIZE)\
                 .prefetch(AUTOTUNE)

test_ds  = test_ds.batch(BATCH_SIZE)\
                  .prefetch(AUTOTUNE)

print("Ready:", train_ds, val_ds, test_ds)

Ready: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))> <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))> <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


## Q3 — Utilities: Compile / Train / Evaluate / Count Params

### Student Tasks

- Compile the model using the Adam optimizer, sparse categorical cross-entropy loss, and accuracy metric.

- Train the model for several epochs and measure the training time.

- Evaluate the trained model on the validation and test datasets and report accuracy.

In [6]:
# ============================================================
# Question Q3 — Model Training Utilities (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Compile a model using the Adam optimizer
# 2) Define the loss function for multi-class classification
# 3) Track accuracy during training
# 4) Compute total model parameters
# 5) Train the model and measure training time
# 6) Evaluate the model on validation and test datasets
# ============================================================

import numpy as np
import time
import tensorflow as tf

# ------------------------------------------------------------
# TODO 1: Compile the model
# ------------------------------------------------------------
def compile_model(model, lr=1e-3):

    model.compile(

        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),

        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),

        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# TODO 2: Compute total parameters of the model
# ------------------------------------------------------------
def total_params(model):

    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights])) + \
           int(np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))


# ------------------------------------------------------------
# TODO 3: Train the model and measure training time
# ------------------------------------------------------------
def train_and_time(model, run_name, epochs=8):

    t0 = time.time()

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=8,
        verbose=1
    )

    dt = time.time() - t0

    return hist, dt


# ------------------------------------------------------------
# TODO 4: Evaluate the model
# ------------------------------------------------------------
def evaluate(model, name):

    v = model.evaluate(val_ds, verbose=0)

    t = model.evaluate(test_ds, verbose=0)

    print(f"{name} | Val acc: {v[1]:.4f} | Test acc: {t[1]:.4f}")

    return {
        "val_loss": v[0],
        "val_acc": v[1],
        "test_loss": t[0],
        "test_acc": t[1],
    }

## Q4 — Backbones & Transfer Model Builder

We will compare three pretrained ImageNet backbones using the **same head design** (GAP + Dropout + Dense):
- **VGG16**
- **ResNet50**
- **DenseNet121**

For feature extraction, keep the backbone **frozen** (`train_base=False`). For fine-tuning, unfreeze a small number of top layers with a smaller learning rate.

### Student Tasks

- Implement a function to load a pretrained backbone model (VGG16, ResNet50, or DenseNet121).

- Build a transfer learning model by adding a Global Average Pooling and classification layer.

- Optionally unfreeze the last layers of the backbone to perform fine-tuning with a smaller learning rate.


In [7]:
# ============================================================
# Question Q4 — Transfer Learning Model Construction (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a backbone model using pretrained ImageNet weights
# 2) Support VGG16, ResNet50, and DenseNet121 architectures
# 3) Attach a classifier head using Global Average Pooling
# 4) Create a full transfer learning model
# 5) Implement fine-tuning by unfreezing part of the backbone
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build backbone model
# ------------------------------------------------------------
def build_backbone(backbone_name: str):
    """Return (base_model, preprocess_fn) for a supported backbone."""

    name = backbone_name.lower()

    if name == "vgg16":
        base = tf.keras.applications.VGG16(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = vgg16.preprocess_input

    elif name == "resnet50":
        base = tf.keras.applications.ResNet50(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = resnet.preprocess_input

    elif name in ["densenet121", "dense121"]:
        base = tf.keras.applications.DenseNet121(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = densenet.preprocess_input

    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    return base, preprocess_fn


# ------------------------------------------------------------
# TODO 2: Build transfer learning model
# ------------------------------------------------------------
def build_transfer_model(backbone_name="resnet50", train_base=False, dropout=0.2):
    """Backbone + GAP head. Returns (model, base)."""

    base, preprocess_fn = build_backbone(backbone_name)

    # Freeze or unfreeze backbone
    base.trainable = bool(train_base)

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

    # Apply preprocessing function
    x = preprocess_fn(inputs * 225.0)

    # Forward pass through backbone
    x = base(x, training=train_base)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)

    # Dropout regularization
    x = layers.Dropout(dropout)(x)

    # Output classification layer
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    # Build final model
    model = tf.keras.Model(
        inputs,
        outputs,
        name=f"{backbone_name}_gap_{'ft' if train_base else 'fz'}"
    )

    return model, base


# ------------------------------------------------------------
# TODO 3: Fine-tune the backbone
# ------------------------------------------------------------
def fine_tune(model, base, n_unfreeze=30, lr=1e-5):
    """Unfreeze last layers of backbone and recompile model."""

    # Enable backbone training
    base.trainable = True

    if n_unfreeze is not None:
        for layer in base.layers[:-int(n_unfreeze)]:
            layer.trainable = False

    # Recompile model with smaller learning rate
    model = compile_model(model, lr=lr)

    return model

## Q5 — Model A: **VGG16** Transfer Learning (Frozen Base)
Train only a small classification head first (feature extraction).

### Student Tasks

- Build a transfer learning model using a frozen VGG16 backbone with a Global Average Pooling head.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.

In [8]:
# ============================================================
# Question Q5 — Train VGG16 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the VGG16 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model with the provided compile_model() function
# 4) Train the model and record training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the VGG16 transfer learning model
# ------------------------------------------------------------
vgg_model, vgg_base = build_transfer_model(
    backbone_name="vgg16",
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
vgg_model = compile_model(
    vgg_model,
    lr=1e-3
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
vgg_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_vgg_fz, time_vgg_fz = train_and_time(
    vgg_model,
    "vgg16_frozen",
    epochs=8
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_vgg_fz = evaluate(
    vgg_model,
    "vgg16_frozen"
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "vgg16_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 128, 128,  │          0 │ input_layer_1[0]… │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 128, 128)  │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 128, 128)  │          0 │ multiply[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 128, 128)  │          0 │ multiply[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 128, 128,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 128,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 4, 4, 512) │ 14,714,688 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 37)        │     18,981 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,733,669 (56.20 MB)

 Trainable params: 18,981 (74.14 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 636s 7s/step - accuracy: 0.1233 - loss: 19.1798 - val_accuracy: 0.3682 - val_loss: 7.3669
Epoch 2/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 618s 7s/step - accuracy: 0.3770 - loss: 7.9758 - val_accuracy: 0.5462 - val_loss: 4.4842
Epoch 3/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 619s 7s/step - accuracy: 0.5255 - loss: 4.9530 - val_accuracy: 0.5965 - val_loss: 3.7415
Epoch 4/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 638s 7s/step - accuracy: 0.6121 - loss: 3.5739 - val_accuracy: 0.6467 - val_loss: 3.3432
Epoch 5/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 610s 7s/step - accuracy: 0.6671 - loss: 2.6849 - val_accuracy: 0.6603 - val_loss: 3.0202
Epoch 6/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 609s 7s/step - accuracy: 0.7031 - loss: 2.2606 - val_accuracy: 0.6780 - val_loss: 2.8913
Epoch 7/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 619s 7s/step - accuracy: 0.7476 - loss: 1.8223 - val_accuracy: 0.6834 - val_loss: 2.9537
Epoch 8/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 642s 7s/step - accuracy: 0.7602 - loss: 1.6539 - val_accuracy: 0.6957 - val_loss

## Q6 — Model B: **ResNet50** Transfer Learning (Frozen Base)
Same head design as VGG16 model for a fair backbone comparison.

### Student Tasks

- Build a transfer learning model using a frozen ResNet50 backbone with a Global Average Pooling head.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.

In [9]:
# ============================================================
# Question Q6 — Train ResNet50 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the ResNet50 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model using compile_model()
# 4) Train the model and record training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the ResNet50 transfer learning model
# ------------------------------------------------------------
resnet_model, resnet_base = build_transfer_model(
    backbone_name="resnet50",
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
resnet_model = compile_model(
    resnet_model,
    lr=1e-3
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
resnet_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_resnet_fz, time_resnet_fz = train_and_time(
    resnet_model,
    "resnet50_frozen",
    epochs=8
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_resnet_fz = evaluate(
    resnet_model,
    "resnet50_frozen"
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "resnet50_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 128, 128,  │          0 │ input_layer_3[0]… │
│ (Multiply)          │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 128, 128)  │          0 │ multiply_1[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 128, 128)  │          0 │ multiply_1[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 128, 128)  │          0 │ multiply_1[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 128, 128,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 128, 128,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 4, 4,      │ 23,587,712 │ add_1[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 37)        │     75,813 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,663,525 (90.27 MB)

 Trainable params: 75,813 (296.14 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 198s 2s/step - accuracy: 0.4626 - loss: 2.3280 - val_accuracy: 0.7283 - val_loss: 0.9499
Epoch 2/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 196s 2s/step - accuracy: 0.7700 - loss: 0.7701 - val_accuracy: 0.7840 - val_loss: 0.7937
Epoch 3/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 208s 2s/step - accuracy: 0.8577 - loss: 0.4211 - val_accuracy: 0.7717 - val_loss: 0.8040
Epoch 4/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.9008 - loss: 0.2973 - val_accuracy: 0.7758 - val_loss: 0.8350
Epoch 5/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 189s 2s/step - accuracy: 0.9327 - loss: 0.1904 - val_accuracy: 0.7799 - val_loss: 0.7739
Epoch 6/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.9467 - loss: 0.1516 - val_accuracy: 0.7772 - val_loss: 0.8015
Epoch 7/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - accuracy: 0.9688 - loss: 0.1050 - val_accuracy: 0.8030 - val_loss: 0.7335
Epoch 8/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 182s 2s/step - accuracy: 0.9749 - loss: 0.0860 - val_accuracy: 0.7935 - val_loss:

## Q7 — Model C: **DenseNet121** Transfer Learning (Frozen Base)
Train a DenseNet121 backbone with the same GAP head design for a fair comparison against VGG16 and ResNet50.

### Student Tasks

- Build a transfer learning model using a frozen DenseNet121 backbone.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.


In [10]:
# ============================================================
# Question Q7 — Train DenseNet121 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the DenseNet121 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model using compile_model()
# 4) Train the model and measure training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the DenseNet121 transfer learning model
# ------------------------------------------------------------
densenet_model, densenet_base = build_transfer_model(
    backbone_name="densenet121",    # e.g., densenet121
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
densenet_model = compile_model(
    densenet_model,
    lr=0.2
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
densenet_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_densenet_fz, time_densenet_fz = train_and_time(
    densenet_model,
    "densnet121_frozen",
    epochs=8
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_densenet_fz = evaluate(
    densenet_model,
    "densenet121_frozen"
)

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "densenet121_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply_2 (Multiply)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_2 (Add)                     │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 37)             │        37,925 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,075,429 (26.99 MB)

 Trainable params: 37,925 (148.14 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

Epoch 1/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.5537 - loss: 68.7302 - val_accuracy: 0.7052 - val_loss: 27.9267
Epoch 2/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - accuracy: 0.7588 - loss: 23.6931 - val_accuracy: 0.7446 - val_loss: 29.3632
Epoch 3/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - accuracy: 0.8030 - loss: 19.5982 - val_accuracy: 0.7242 - val_loss: 42.6153
Epoch 4/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 175s 2s/step - accuracy: 0.8434 - loss: 16.8829 - val_accuracy: 0.7582 - val_loss: 34.7864
Epoch 5/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 177s 2s/step - accuracy: 0.8716 - loss: 13.4592 - val_accuracy: 0.7500 - val_loss: 40.3651
Epoch 6/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 201s 2s/step - accuracy: 0.8614 - loss: 14.1772 - val_accuracy: 0.7486 - val_loss: 43.4276
Epoch 7/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 176s 2s/step - accuracy: 0.8645 - loss: 15.4557 - val_accuracy: 0.7649 - val_loss: 45.0251
Epoch 8/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - accuracy: 0.8808 - loss: 14.5419 - val_accuracy: 0.7

## Q8 — Fine-Tuning Experiment (One Controlled Change)

Fine-tune **all three backbone models (VGG16, ResNet50, and DenseNet121)** by unfreezing only the **top N layers**.

Report whether performance **improves, stays similar, or degrades**, and briefly explain why.

**Recommended**
- Start by unfreezing the **last 10–30 layers**
- Use a **smaller learning rate** (e.g., `1e-5`)
- Use **2-3 epochs**

### Student Tasks

- Fine-tune the **VGG16, ResNet50, and DenseNet121** models by unfreezing the top layers of each backbone.

- Train each fine-tuned model for several epochs.

- Evaluate the validation and test accuracy of each fine-tuned model.

## Q8(a) — Fine-tune VGG16

In [11]:
# ============================================================
# Question Q8(a) — Fine-Tune VGG16 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the VGG16 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the VGG16 backbone
# ------------------------------------------------------------
vgg_model = fine_tune(
    vgg_model,
    vgg_base,
    n_unfreeze=30,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_vgg_ft, time_vgg_ft = train_and_time(
    vgg_model,
    "vgg16finetuned",
    epochs=8
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_vgg_ft = evaluate(
    vgg_model,
    "vgg16fine_tuned"
)

Epoch 1/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 1913s 21s/step - accuracy: 0.7554 - loss: 1.0181 - val_accuracy: 0.6821 - val_loss: 1.3766
Epoch 2/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 1881s 20s/step - accuracy: 0.8461 - loss: 0.5057 - val_accuracy: 0.7147 - val_loss: 1.2715
Epoch 3/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 1878s 20s/step - accuracy: 0.8849 - loss: 0.3731 - val_accuracy: 0.7160 - val_loss: 1.2856
Epoch 4/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 1879s 20s/step - accuracy: 0.9130 - loss: 0.2690 - val_accuracy: 0.7188 - val_loss: 1.2467
Epoch 5/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 1907s 21s/step - accuracy: 0.9300 - loss: 0.2103 - val_accuracy: 0.7188 - val_loss: 1.2355
Epoch 6/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 1912s 21s/step - accuracy: 0.9477 - loss: 0.1614 - val_accuracy: 0.7310 - val_loss: 1.2724
Epoch 7/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 1886s 20s/step - accuracy: 0.9603 - loss: 0.1406 - val_accuracy: 0.7310 - val_loss: 1.2546
Epoch 8/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 1884s 20s/step - accuracy: 0.9718 - loss: 0.1011 - val_accuracy: 0.

## Q8(b) — Fine-tune ResNet50

In [12]:
# ============================================================
# Question Q8(b) — Fine-Tune ResNet50 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the ResNet50 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the ResNet50 backbone
# ------------------------------------------------------------
resnet_model = fine_tune(
    resnet_model,
    resnet_base,
    n_unfreeze=30,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_resnet_ft, time_resnet_ft = train_and_time(
    resnet_model,
    "resnet_fine_tuned",
    epochs=8
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_resnet_ft = evaluate(
    resnet_model,
    "renet_fine_tuned"
)

Epoch 1/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 329s 3s/step - accuracy: 0.9032 - loss: 0.4417 - val_accuracy: 0.7826 - val_loss: 0.7291
Epoch 2/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 319s 3s/step - accuracy: 0.9667 - loss: 0.2446 - val_accuracy: 0.7731 - val_loss: 0.7530
Epoch 3/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 310s 3s/step - accuracy: 0.9857 - loss: 0.1723 - val_accuracy: 0.7717 - val_loss: 0.7652
Epoch 4/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 309s 3s/step - accuracy: 0.9895 - loss: 0.1316 - val_accuracy: 0.7717 - val_loss: 0.7681
Epoch 5/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 318s 3s/step - accuracy: 0.9956 - loss: 0.0967 - val_accuracy: 0.7772 - val_loss: 0.7559
Epoch 6/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 320s 3s/step - accuracy: 0.9963 - loss: 0.0831 - val_accuracy: 0.7772 - val_loss: 0.7476
Epoch 7/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 304s 3s/step - accuracy: 0.9986 - loss: 0.0648 - val_accuracy: 0.7772 - val_loss: 0.7353
Epoch 8/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 320s 3s/step - accuracy: 0.9969 - loss: 0.0580 - val_accuracy: 0.7853 - val_loss:

## Q8(c) — Fine-tune DenseNet121

In [13]:
# ============================================================
# Question Q8(c) — Fine-Tune DenseNet121 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the DenseNet121 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the DenseNet121 backbone
# ------------------------------------------------------------
densenet_model = fine_tune(
    densenet_model,
    densenet_base,
    n_unfreeze=30,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_densenet_ft, time_densenet_ft = train_and_time(
    densenet_model,
    "densenet121_fine_tuned",
    epochs=8
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_densenet_ft = evaluate(
    densenet_model,
    "densenet121_fine_tuned"
)

Epoch 1/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 207s 2s/step - accuracy: 0.8597 - loss: 11.0991 - val_accuracy: 0.7785 - val_loss: 31.3645
Epoch 2/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 187s 2s/step - accuracy: 0.8682 - loss: 10.0346 - val_accuracy: 0.7826 - val_loss: 29.0027
Epoch 3/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 192s 2s/step - accuracy: 0.8736 - loss: 9.2738 - val_accuracy: 0.7867 - val_loss: 27.9559
Epoch 4/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 178s 2s/step - accuracy: 0.8815 - loss: 8.0845 - val_accuracy: 0.7989 - val_loss: 27.1363
Epoch 5/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.8944 - loss: 7.3287 - val_accuracy: 0.8016 - val_loss: 26.5110
Epoch 6/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.8879 - loss: 7.5302 - val_accuracy: 0.8016 - val_loss: 25.9037
Epoch 7/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.8978 - loss: 6.0001 - val_accuracy: 0.8057 - val_loss: 25.3562
Epoch 8/8
92/92 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.9076 - loss: 5.6789 - val_accuracy: 0.8139 - 

## Q9 — Performance Comparison Table

#### Create a compact table comparing **frozen-backbone** models:

- VGG16 (frozen)
- ResNet50 (frozen)
- DenseNet121 (frozen)

#### Also include **fine-tuned** results for all three backbone.

### Student Tasks

- Create a comparison table summarizing the results of all models.

- Include validation accuracy, test accuracy, and training time.

- Compare frozen and fine-tuned models and identify the best-performing model.


In [15]:
# ============================================================
# Question Q9 — Compare Model Results (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Store evaluation results for frozen models
# 2) Append results for fine-tuned models if they exist
# 3) Create a comparison table using pandas
# 4) Sort models by test accuracy
# ============================================================

# ------------------------------------------------------------
# TODO 1: Create result rows for frozen models
# ------------------------------------------------------------
rows = [

    {"Model": "VGG16 Frozen",
     "Val acc": res_vgg_fz["val_acc"],
     "Test acc": res_vgg_fz["test_acc"],
     "Train time (s)": time_vgg_fz},

    {"Model": "ResNet50 Frozen",
     "Val acc": res_resnet_fz["val_acc"],
     "Test acc": res_resnet_fz["test_acc"],
     "Train time (s)": time_resnet_fz},

    {"Model": "DenseNet121 Frozen",
     "Val acc": res_densenet_fz["val_acc"],
     "Test acc": res_densenet_fz["test_acc"],
     "Train time (s)": time_densenet_fz},
]

# ------------------------------------------------------------
# TODO 2: Append results for fine-tuned models if available
# ------------------------------------------------------------

if "res_vgg_ft" in globals():
    rows.append({
        "Model": "VGG16 Fine-Tuned",
        "Val acc": res_vgg_ft["val_acc"],
        "Test acc": res_vgg_ft["test_acc"],
        "Train time (s)": time_vgg_ft
    })

if "res_resnet_ft" in globals():
    rows.append({
        "Model": "ResNet50 Fine-Tuned",
        "Val acc": res_resnet_ft["val_acc"],
        "Test acc": res_resnet_ft["test_acc"],
        "Train time (s)": time_resnet_ft
    })

if "res_densenet_ft" in globals():
    rows.append({
        "Model": "DenseNet121 Fine-Tuned",
        "Val acc": res_densenet_ft["val_acc"],
        "Test acc": res_densenet_ft["test_acc"],
        "Train time (s)": time_densenet_ft
    })


# ------------------------------------------------------------
# TODO 3: Create comparison dataframe
# ------------------------------------------------------------
df = pd.DataFrame(rows)


# ------------------------------------------------------------
# TODO 4: Sort models by test accuracy
# ------------------------------------------------------------
df = df.sort_values("Test acc", ascending=False)

df

,Model,Val acc,Test acc,Train time (s)
5,DenseNet121 Fine-Tuned,0.813859,0.769965,1541.451329
2,DenseNet121 Frozen,0.777174,0.760970,1485.902747
1,ResNet50 Frozen,0.793478,0.751976,1554.174764
4,ResNet50 Fine-Tuned,0.785326,0.742982,2549.978551
3,VGG16 Fine-Tuned,0.726902,0.704552,15140.812242
0,VGG16 Frozen,0.695652,0.663123,4990.593646


## Short Written Questions

## **Q10** — What is the difference between *feature extraction* and *fine-tuning* in transfer learning?  
**Answer:**
Feature extraction uses pre-trained neural networks as a fixed feature extractor where the backbone layers are froze. The only new added classification layers are trained on the target database which requires less computation and reduces the risk of overfitting when the data set is small. Fine tuning unfreezes some or all of the pre-trained backbone layers and continues training them along with the other classifiers. This allows the model to adapt the learned features to the new data set which improves accuracy. In this code, VGG16, ResNet50, and DenseNet121 backbones were initially frozen and then later on the last layers were unfrozen and trained witha smaller learning rate.

## **Q11** — Why should the learning rate typically be lower during fine-tuning than during feature extraction?  

**Answer:**
The learning rate should be lower becuase it allows the model to make small, gradual adjustments to the pre-trained weights whie preseving useful knowledge that was already learned. A lower learning rate during fine tuning prevents large changes to pre-trained weights, reducing the risk of catastropihc forgetting and can lead to more stable straining and better model performance.

## **Q12** — Give one reason why these backbones may behave differently on Oxford-IIIT Pet.  

**Answer:** One reason these backbones may behave differently is that they have different neural network arcitechtures and capacities. VGG116 uses a simple sequential architecture with many parameters. ResNet50 uses skipping connections that help train deeper networks. DenseNet121 uses dense connections that encourage feature reuse and more efficient information flow between laters. These architectual differnces allow each model to extract different visual features from pet images and may generalize differently to the Oxfor-IIIT Pet dataset.

## **Q13** — Under what conditions can fine-tuning reduce performance compared to a frozen backbone?  

**Answer:**
Fine-tuning can reduce performane when the model overfits the training data or when the pre-trained weights are modified too aggressively. This is more likely when the target data set is small, the learning rate is too high, or when too many layers are frozen. A high learning rate can cause large weight updates that destroy useful information learned from ImageNet, leading to catastrophic forgetting. Fine-tuning can perform worse than frozen backbones when it causes over fitting or catastrophic forgetting.

### 🎉 Congratulations!

You have successfully completed **P4-TL**. Excellent work exploring and comparing **RL**, specifically **VGG16**, **VGG16**, and **DenseNet121**. Analyzing how **TL, and fine-tuning** affect performance on the **Oxford-IIIT Pet** under **CPU-only training constraints**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q9)** have been executed successfully
- All **Markdown responses (Q10–Q13)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.